# Cohort Retention
Cohort adalah cara mengelompokkan pelanggan berdasarkan kapan mereka pertama kali belanja (misal: “cohort Januari 2017” = semua pelanggan yang order pertamanya di bulan itu), lalu kita lihat berapa persen dari grup itu yang kembali belanja lagi di bulan-bulan berikutnya. Ini menjawab pertanyaan “apakah pelanggan baru cenderung balik lagi, atau cuma belanja sekali lalu hilang?”

**Insight:**

Tabel retensi menunjukkan pola yang sangat konsisten di seluruh cohort 
(Januari 2017 - Agustus 2018): retensi bulan ke-1 berkisar antara 
0.18% - 0.72%, dan terus berada di bawah 1% di bulan-bulan berikutnya, 
tanpa pengecualian berarti pada cohort manapun (cohort dengan ukuran 
sampel sangat kecil seperti 2016-09 dan 2016-12 dikecualikan karena 
tidak representatif).

Sebagai contoh, cohort terbesar (2017-11, dengan 7,060 pelanggan baru) 
hanya menyisakan 0.57% pelanggan yang kembali bertransaksi di bulan 
berikutnya. Pola rendahnya retensi ini konsisten di seluruh periode 
data, dan selaras dengan temuan RFM sebelumnya bahwa lebih dari 75% 
pelanggan Olist tercatat hanya melakukan satu kali transaksi.

In [3]:
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    host="localhost",
    dbname="Olist Ecommerce Database",
    user="postgres",
    password="admin123",
    port=5432
)

query_cohort = """
SELECT 
    c.customer_unique_id,
    o.order_id,
    o.order_purchase_timestamp
FROM olist_orders_dataset o
JOIN olist_customers_dataset c ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered'
"""

df_cohort = pd.read_sql(query_cohort, conn)
conn.close()

df_cohort.head()

C:\Users\alex7\AppData\Local\Temp\ipykernel_240\753735017.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_cohort = pd.read_sql(query_cohort, conn)


,customer_unique_id,order_id,order_purchase_timestamp
0,7c396fd4830fd04220f754e42b4e5bff,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33
1,72632f0f9dd73dfee390c9b22eb56dd6,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39
2,80bb27c7c16e8f973207a5086ab329e2,a4591c265e18cb1dcee52889e2d8acc3,2017-07-09 21:57:05
3,39382392765b6dc74812866ee5ee92a7,76c6e866289321a7c93b82b54852dc33,2017-01-23 18:29:09
4,e97109680b052ee858d93a539597bba7,82566a660a982b15fb86e904c8d32918,2018-06-07 10:06:19


In [4]:
# Bulan dari tiap transaksi (dibulatkan ke awal bulan)
df_cohort['order_month'] = df_cohort['order_purchase_timestamp'].dt.to_period('M')

# Bulan cohort = bulan transaksi PERTAMA tiap pelanggan
df_cohort['cohort_month'] = df_cohort.groupby('customer_unique_id')['order_month'].transform('min')

df_cohort.head()

,customer_unique_id,order_id,order_purchase_timestamp,order_month,cohort_month
0,7c396fd4830fd04220f754e42b4e5bff,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,2017-10,2017-09
1,72632f0f9dd73dfee390c9b22eb56dd6,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39,2018-02,2018-02
2,80bb27c7c16e8f973207a5086ab329e2,a4591c265e18cb1dcee52889e2d8acc3,2017-07-09 21:57:05,2017-07,2017-07
3,39382392765b6dc74812866ee5ee92a7,76c6e866289321a7c93b82b54852dc33,2017-01-23 18:29:09,2017-01,2017-01
4,e97109680b052ee858d93a539597bba7,82566a660a982b15fb86e904c8d32918,2018-06-07 10:06:19,2018-06,2018-06


In [5]:
# Hitung selisih bulan antara order_month dan cohort_month
# cohort_index = 0 artinya bulan pertama kali belanja (cohort itu sendiri)
# cohort_index = 1 artinya 1 bulan setelah pertama kali belanja, dst
df_cohort['cohort_index'] = (
    (df_cohort['order_month'].dt.year - df_cohort['cohort_month'].dt.year) * 12 +
    (df_cohort['order_month'].dt.month - df_cohort['cohort_month'].dt.month)
)

df_cohort.head()

,customer_unique_id,order_id,order_purchase_timestamp,order_month,cohort_month,cohort_index
0,7c396fd4830fd04220f754e42b4e5bff,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,2017-10,2017-09,1
1,72632f0f9dd73dfee390c9b22eb56dd6,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39,2018-02,2018-02,0
2,80bb27c7c16e8f973207a5086ab329e2,a4591c265e18cb1dcee52889e2d8acc3,2017-07-09 21:57:05,2017-07,2017-07,0
3,39382392765b6dc74812866ee5ee92a7,76c6e866289321a7c93b82b54852dc33,2017-01-23 18:29:09,2017-01,2017-01,0
4,e97109680b052ee858d93a539597bba7,82566a660a982b15fb86e904c8d32918,2018-06-07 10:06:19,2018-06,2018-06,0


In [6]:
cohort_data = df_cohort.groupby(['cohort_month', 'cohort_index'])['customer_unique_id'].nunique().reset_index()

# Pivot supaya cohort_month jadi baris, cohort_index jadi kolom
cohort_pivot = cohort_data.pivot(index='cohort_month', columns='cohort_index', values='customer_unique_id')

cohort_pivot

cohort_index,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,19,20
cohort_month,,,,,,,,,,,,,,,,,,,,
2016-09,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2016-10,262.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,1.0,NaN,1.0,NaN,1.0,NaN,1.0,NaN,1.0,2.0,2.0
2016-12,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-01,717.0,2.0,2.0,1.0,3.0,1.0,3.0,1.0,1.0,NaN,3.0,1.0,5.0,3.0,1.0,1.0,2.0,3.0,1.0,NaN
2017-02,1628.0,3.0,5.0,2.0,7.0,2.0,4.0,3.0,2.0,3.0,2.0,5.0,2.0,3.0,2.0,1.0,1.0,3.0,NaN,NaN
2017-03,2503.0,11.0,9.0,10.0,9.0,4.0,4.0,8.0,8.0,2.0,9.0,3.0,5.0,3.0,4.0,6.0,2.0,3.0,NaN,NaN
2017-04,2256.0,14.0,5.0,4.0,6.0,6.0,8.0,7.0,7.0,4.0,6.0,2.0,1.0,1.0,2.0,2.0,3.0,NaN,NaN,NaN
2017-05,3451.0,16.0,16.0,10.0,10.0,11.0,14.0,5.0,9.0,9.0,9.0,12.0,8.0,1.0,6.0,7.0,NaN,NaN,NaN,NaN
2017-06,3037.0,15.0,12.0,13.0,9.0,12.0,11.0,7.0,4.0,6.0,9.0,11.0,5.0,5.0,7.0,NaN,NaN,NaN,NaN,NaN


In [7]:
# Ambil ukuran awal tiap cohort (kolom index 0)
cohort_size = cohort_pivot.iloc[:, 0]

# Bagi tiap kolom dengan ukuran awal cohort-nya masing-masing, kalikan 100
retention_table = cohort_pivot.divide(cohort_size, axis=0) * 100

retention_table.round(2)

cohort_index,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,19,20
cohort_month,,,,,,,,,,,,,,,,,,,,
2016-09,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2016-10,100.0,NaN,NaN,NaN,NaN,NaN,0.38,NaN,NaN,0.38,NaN,0.38,NaN,0.38,NaN,0.38,NaN,0.38,0.76,0.76
2016-12,100.0,100.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-01,100.0,0.28,0.28,0.14,0.42,0.14,0.42,0.14,0.14,NaN,0.42,0.14,0.70,0.42,0.14,0.14,0.28,0.42,0.14,NaN
2017-02,100.0,0.18,0.31,0.12,0.43,0.12,0.25,0.18,0.12,0.18,0.12,0.31,0.12,0.18,0.12,0.06,0.06,0.18,NaN,NaN
2017-03,100.0,0.44,0.36,0.40,0.36,0.16,0.16,0.32,0.32,0.08,0.36,0.12,0.20,0.12,0.16,0.24,0.08,0.12,NaN,NaN
2017-04,100.0,0.62,0.22,0.18,0.27,0.27,0.35,0.31,0.31,0.18,0.27,0.09,0.04,0.04,0.09,0.09,0.13,NaN,NaN,NaN
2017-05,100.0,0.46,0.46,0.29,0.29,0.32,0.41,0.14,0.26,0.26,0.26,0.35,0.23,0.03,0.17,0.20,NaN,NaN,NaN,NaN
2017-06,100.0,0.49,0.40,0.43,0.30,0.40,0.36,0.23,0.13,0.20,0.30,0.36,0.16,0.16,0.23,NaN,NaN,NaN,NaN,NaN


In [8]:
# Export tabel retensi ke CSV untuk dipakai di Power BI
# reset_index() supaya cohort_month jadi kolom biasa, bukan index
retention_export = retention_table.round(2).reset_index()
retention_export.to_csv('../data/processed/retention_table.csv', index=False)
print("Berhasil disimpan ke data/processed/retention_table.csv")

Berhasil disimpan ke data/processed/retention_table.csv
